# Conditional object IoU fallback on VALID
Attach the four previous replay datasets and mscoco-2014 for original images. Accelerator None. Output: csa_relation_fallback_valid_review.zip.

In [ ]:
SOURCES={'prompt_adapter_data.py': '"""NumPy-only instance joins for the region-linked CSA prompt pilot."""\nfrom collections import Counter\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\n\nPROMPT_SHA = \'24e7861a4aa0e7e731d67689dce195688096232bcbef1b4264034cf520d6957d\'\nOBJECT_CAP, RELATION_CAP, IOU = 8, 3, .5\n\n\ndef label(value):\n    if not isinstance(value, str) or not value.strip():\n        raise ValueError(\'Missing object label\')\n    return \' \'.join(value.lower().replace(\'_\', \' \').split())\n\n\ndef box(value):\n    b = np.asarray(value, dtype=np.float64)\n    if b.shape != (4,) or not np.isfinite(b).all() or (b < 0).any() or (b > 1).any() or (b[2:] <= b[:2]).any():\n        raise ValueError(\'Expected normalized positive xyxy box\')\n    return b\n\n\ndef overlap(a, boxes):\n    a = box(a)\n    if not len(boxes):\n        return np.empty(0)\n    boxes = np.stack([box(b) for b in boxes])\n    intersection = np.maximum(0, np.minimum(a[2:], boxes[:, 2:]) - np.maximum(a[:2], boxes[:, :2])).prod(1)\n    return intersection / ((a[2:] - a[:2]).prod() + (boxes[:, 2:] - boxes[:, :2]).prod(1) - intersection)\n\n\ndef detector_regions(objects, width, height, saved_boxes, kinds):\n    """Reconstruct the existing cache\'s confidence-sort/clip/cap before label joins."""\n    if width <= 0 or height <= 0 or len(kinds) != len(saved_boxes):\n        raise ValueError(\'Invalid ROI dimensions\')\n    ordered = sorted(enumerate(objects), key=lambda x: (-float(x[1][\'conf\']), x[0]))[:150]\n    if not ordered:\n        if kinds != [\'full_frame_fallback\'] or not np.allclose(saved_boxes, [[0, 0, width, height]], rtol=0, atol=1e-4):\n            raise ValueError(\'Changed fallback ROI\')\n        return []\n    if kinds != [\'predicted_roi\'] * len(ordered):\n        raise ValueError(\'Detector and ROI counts/kinds disagree\')\n    result = []\n    for i, (_, obj) in enumerate(ordered):\n        score = float(obj[\'conf\'])\n        if not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid detector confidence\')\n        b = np.clip(np.asarray(obj[\'bbox\'], np.float32), [0, 0, 0, 0], [width, height, width, height]).astype(np.float32)\n        if b.shape != (4,) or not np.isfinite(b).all() or not np.allclose(b, saved_boxes[i], atol=1e-4, rtol=0):\n            raise ValueError(\'Detector order/box differs from saved ROI\')\n        result.append(dict(roi=i, label=label(obj[\'label\']), bbox=box(b / [width, height, width, height]).tolist()))\n    return result\n\n\ndef geometry(subject, obj):\n    s, o = box(subject), box(obj)\n    sw, sh = s[2:] - s[:2]; ow, oh = o[2:] - o[:2]\n    sc, oc = (s[:2] + s[2:]) / 2, (o[:2] + o[2:]) / 2\n    intersection = np.maximum(0, np.minimum(s[2:], o[2:]) - np.maximum(s[:2], o[:2])).prod()\n    return [float(v) for v in ((oc[0]-sc[0])/sw, (oc[1]-sc[1])/sh, np.log(ow/sw), np.log(oh/sh),\n            intersection/(sw*sh+ow*oh-intersection), intersection/(sw*sh), intersection/(ow*oh), np.linalg.norm(oc-sc))]\n\n\ndef prompt_text(name):\n    # Use the established UPT article handling, including plural skis/scissors.\n    from captioning.upt_prompts import noun\n    return noun(name)\n\n\ndef join_entry(entry, regions):\n    """Only selected objects/relations are consumed; previews/refs/GT are ignored."""\n    if entry[\'mode\'] not in (\'object_relation\', \'object_only\', \'no_objects\'):\n        raise ValueError(\'Unknown prompt mode\')\n    objects = entry[\'objects\']; relations = entry[\'relations\']\n    if not isinstance(objects, list) or not isinstance(relations, list) or len(relations) > RELATION_CAP:\n        raise ValueError(\'Malformed or unapproved prompt caps\')\n    by_index = {}; instances = set(); mapped = {}; stats = Counter(objects=len(objects), relations=len(relations))\n    for obj in objects:\n        index = obj[\'index\']; instance = obj[\'instance_id\']; name = label(obj[\'label\']); b = box(obj[\'bbox\'])\n        score = float(obj[\'confidence\'])\n        if type(index) != int or index in by_index or not isinstance(instance, str) or not instance or instance in instances:\n            raise ValueError(\'Duplicate/invalid object identity\')\n        if not np.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid UPT object score\')\n        by_index[index] = obj; instances.add(instance)\n        candidates = [r for r in regions if r[\'label\'] == name]\n        values = overlap(b, [r[\'bbox\'] for r in candidates])\n        if not len(values) or values.max() < IOU:\n            stats[\'unmatched_objects\'] += 1; continue\n        best = np.flatnonzero(np.isclose(values, values.max(), rtol=0, atol=1e-8))\n        if len(best) != 1:\n            stats[\'ambiguous_objects\'] += 1; continue\n        j = int(best[0]); r = candidates[j]\n        mapped[index] = dict(instance_id=instance, upt_index=index, roi=r[\'roi\'], label=name, bbox=b.tolist(),\n            roi_bbox=r[\'bbox\'], IoU=float(values[j]), confidence=score, text=prompt_text(name))\n    collisions = Counter(r[\'roi\'] for r in mapped.values())\n    for index in list(mapped):\n        if collisions[mapped[index][\'roi\']] > 1:\n            del mapped[index]; stats[\'colliding_objects\'] += 1\n    endpoints = []; valid_relations = []; seen_pairs = set()\n    for r in relations:\n        s, o = r[\'subject_index\'], r[\'object_index\']\n        if type(s) != int or type(o) != int or s == o or s not in by_index or o not in by_index:\n            raise ValueError(\'Broken relation endpoints\')\n        if r[\'subject_instance\'] != by_index[s][\'instance_id\'] or r[\'object_instance\'] != by_index[o][\'instance_id\']:\n            raise ValueError(\'Relation instance IDs differ\')\n        if label(r[\'subject\']) != \'person\' or label(by_index[s][\'label\']) != \'person\' or label(r[\'object\']) != label(by_index[o][\'label\']):\n            raise ValueError(\'Wrong subject/object role labels\')\n        score, threshold = float(r[\'score\']), float(r[\'threshold\'])\n        if not np.isfinite([score, threshold]).all() or not 0 <= score <= 1 or not 0 <= threshold <= 1.01 or score < threshold:\n            raise ValueError(\'Rejected/nonfinite relation in selected list\')\n        from captioning.upt_prompts import relation_text\n        expected = relation_text(r[\'predicate\'], label(r[\'object\']))\n        if r[\'text\'] != expected:\n            raise ValueError(\'Relation text differs from locked UPT template\')\n        key = (s, r[\'predicate\'], o)\n        if key in seen_pairs:\n            raise ValueError(\'Duplicate relation\')\n        seen_pairs.add(key)\n        if s not in mapped or o not in mapped:\n            stats[\'unmatched_relations\'] += 1; continue\n        endpoints.extend((s, o))\n        valid_relations.append(dict(subject_instance=r[\'subject_instance\'], object_instance=r[\'object_instance\'],\n            subject_roi=mapped[s][\'roi\'], object_roi=mapped[o][\'roi\'], text=expected, predicate=r[\'predicate\'],\n            score=score, threshold=threshold, geometry=geometry(by_index[s][\'bbox\'], by_index[o][\'bbox\'])))\n    order = list(dict.fromkeys(endpoints))\n    order += [i for i in sorted(mapped, key=lambda i: (-mapped[i][\'confidence\'], i)) if i not in order]\n    selected = [mapped[i] for i in order[:OBJECT_CAP]]\n    stats.update(mapped_objects=len(mapped), kept_objects=len(selected), mapped_relations=len(valid_relations),\n                 capped_objects=max(0, len(mapped)-OBJECT_CAP))\n    return dict(source_mode=entry[\'mode\'], objects=selected, relations=valid_relations, counts=dict(stats))\n\n\ndef cohort(ids):\n    ids = [int(i) for i in ids]\n    if len(ids) != len(set(ids)) or len(ids) < 10000:\n        raise ValueError(\'Need unique Karpathy TRAIN IDs\')\n    return sorted(ids, key=lambda i: hashlib.sha256(f\'42:{i}\'.encode()).hexdigest())[:10000]\n\n\ndef inference_record(meta, linked):\n    """Do not copy captions, sequences or any supervision into prompt payloads."""\n    return dict(coco_id=int(meta[\'coco_id\']), filename=meta[\'filename\'], group=meta[\'group\'],\n                roi_count=len(meta[\'region_kinds\']), **linked)\n\n\ndef tensor_batch(records, embeddings, texts):\n    """Data-only packing shared by the future worker and neural smoke tests."""\n    lookup = {s: i for i, s in enumerate(texts)}\n    if len(lookup) != len(texts) or embeddings.shape != (len(texts), 512) or not np.isfinite(embeddings).all():\n        raise ValueError(\'Invalid prompt embedding cache\')\n    n = len(records)\n    out = dict(object_text=np.zeros((n,8,512),np.float32), object_roi=np.zeros((n,8),np.int64),\n        object_box=np.zeros((n,8,4),np.float32), object_valid=np.zeros((n,8),bool),\n        relation_text=np.zeros((n,3,512),np.float32), relation_pair=np.zeros((n,3,2),np.int64),\n        relation_geometry=np.zeros((n,3,8),np.float32), relation_valid=np.zeros((n,3),bool))\n    for i, row in enumerate(records):\n        if len(row[\'objects\']) > 8 or len(row[\'relations\']) > 3:\n            raise ValueError(\'Prompt slot overflow\')\n        for j, obj in enumerate(row[\'objects\']):\n            if not 0 <= obj[\'roi\'] < row[\'roi_count\']:\n                raise ValueError(\'Object ROI out of bounds\')\n            out[\'object_text\'][i,j]=embeddings[lookup[obj[\'text\']]];out[\'object_roi\'][i,j]=obj[\'roi\']\n            out[\'object_box\'][i,j]=box(obj[\'bbox\']);out[\'object_valid\'][i,j]=True\n        for j, rel in enumerate(row[\'relations\']):\n            pair=[rel[\'subject_roi\'],rel[\'object_roi\']]\n            if pair[0] == pair[1] or any(not 0 <= k < row[\'roi_count\'] for k in pair):\n                raise ValueError(\'Invalid relation pair\')\n            g=np.asarray(rel[\'geometry\'],np.float32)\n            if g.shape!=(8,) or not np.isfinite(g).all():raise ValueError(\'Invalid pair geometry\')\n            out[\'relation_text\'][i,j]=embeddings[lookup[rel[\'text\']]];out[\'relation_pair\'][i,j]=pair\n            out[\'relation_geometry\'][i,j]=g;out[\'relation_valid\'][i,j]=True\n    return out\n', 'captioning/upt_prompts.py': '"""UPT predictions -> instance-linked text; no annotations or reference captions."""\nimport math\n\nfrom .selected_relation_prompts import entity_id, phrase\nfrom .upt_action_text import ACTION_TEXT, action_predicate\n\nMETHOD = \'upt_object_relation_prompt_v1\'\nPREDICATES = (\'holding\', \'riding\')\n\n\ndef noun(label):\n    return label if label in (\'skis\', \'scissors\') else phrase(label)\n\n\ndef relation_text(predicate, obj):\n    verb = {\'holding\': \'hold\', \'riding\': \'ride\'}.get(predicate, predicate)\n    if verb not in ACTION_TEXT:\n        raise ValueError(\'Unsupported prompt predicate.\')\n    # Do not infer surfing/skiing from object class alone. Keep UPT\'s broad ride.\n    if predicate == \'riding\' and obj == \'skis\':\n        return \'a person on skis\'\n    if verb in (\'jump\', \'run\'):\n        # These labels do not specify the object\'s grammatical role; do not invent one.\n        return f\'a person and {noun(obj)} (interaction: {ACTION_TEXT[verb]})\'\n    return f\'a person {ACTION_TEXT[verb]} {noun(obj)}\'\n\n\ndef from_audit(row):\n    """Allowlist inference fields; deliberately discard every GT/matched ID/label."""\n    detections = [{k: d[k] for k in (\'label\', \'confidence\', \'box\')} for d in row[\'detection_details\']]\n    claims = [{k: c[k] for k in (\'subject\', \'object\', \'predicate\', \'score\')} for c in row[\'candidates\']]\n    return detections, claims\n\n\ndef from_cache(row, vocabulary, all_actions=False):\n    h, w = row[\'resized_size_hw\']\n    if h <= 0 or w <= 0 or len(row[\'boxes_resized_xyxy\']) != len(row[\'labels\']) or len(row[\'labels\']) != len(row[\'object_scores\']):\n        raise ValueError(\'Invalid cache detection dimensions.\')\n    detections = []\n    for box, label, score in zip(row[\'boxes_resized_xyxy\'], row[\'labels\'], row[\'object_scores\']):\n        if not 0 <= label < len(vocabulary[\'objects\']):\n            raise ValueError(\'Invalid cache object label.\')\n        detections.append({\'label\': vocabulary[\'objects\'][label].replace(\'_\', \' \'), \'confidence\': score,\n                           \'box\': [max(0., min(1., x / scale)) for x, scale in zip(box, (w, h, w, h))]})\n    if len(row[\'pairs\']) != len(row[\'native_scores\']):\n        raise ValueError(\'Invalid native score dimensions.\')\n    claims = []\n    for (s, o), scores in zip(row[\'pairs\'], row[\'native_scores\']):\n        if not 0 <= s < len(detections) or not 0 <= o < len(detections):\n            raise ValueError(\'Invalid cache pair indices.\')\n        verbs = vocabulary[\'verbs_by_object\'][row[\'labels\'][o]]\n        if len(scores) != len(verbs):\n            raise ValueError(\'Sparse native score row differs from ontology.\')\n        for verb, score in zip(verbs, scores):\n            name = vocabulary[\'verbs\'][verb]\n            if name == \'no_interaction\':\n                continue\n            if all_actions and name not in ACTION_TEXT:\n                raise ValueError(f\'Unknown UPT action: {name}\')\n            if all_actions or name in (\'hold\', \'ride\'):\n                claims.append({\'subject\': s, \'object\': o, \'predicate\': action_predicate(name),\n                               \'score\': score})\n    return detections, claims\n\n\ndef build_prompt(filename, detections, claims, policy, max_relations=3, object_threshold=.5, max_object_names=8):\n    if max_relations < 1 or max_object_names < 1 or not math.isfinite(object_threshold) or not 0 <= object_threshold <= 1:\n        raise ValueError(\'Invalid prompt options.\')\n    thresholds = {p: value[\'threshold\'] for p, value in policy[\'predicates\'].items()}\n    if not thresholds or any(p not in {action_predicate(v) for v in ACTION_TEXT} for p in thresholds):\n        raise ValueError(\'Unknown or empty policy predicates.\')\n    if any(not math.isfinite(t) or not 0 <= t <= 1.01 for t in thresholds.values()):\n        raise ValueError(\'Invalid locked policy threshold.\')\n    objects = []\n    for i, d in enumerate(detections):\n        label, confidence, box = d[\'label\'], d[\'confidence\'], d[\'box\']\n        if not isinstance(label, str) or not label.strip() or label != label.strip():\n            raise ValueError(\'Invalid object name.\')\n        if not math.isfinite(confidence) or not 0 <= confidence <= 1:\n            raise ValueError(\'Invalid object confidence.\')\n        if len(box) != 4 or not all(math.isfinite(v) for v in box) or not 0 <= box[0] < box[2] <= 1 or not 0 <= box[1] < box[3] <= 1:\n            raise ValueError(\'Invalid normalized predicted box.\')\n        objects.append({\'index\': i, \'instance_id\': entity_id(filename, label, box),\n                        \'label\': label, \'confidence\': confidence, \'bbox\': box})\n    unique = {}\n    for c in claims:\n        s, o, p, score = (c[k] for k in (\'subject\', \'object\', \'predicate\', \'score\'))\n        if p not in thresholds or s == o or not 0 <= s < len(objects) or not 0 <= o < len(objects):\n            raise ValueError(\'Invalid relation pair/predicate.\')\n        if objects[s][\'label\'] != \'person\' or not math.isfinite(score) or not 0 <= score <= 1:\n            raise ValueError(\'Invalid relation actor/score.\')\n        key = (s, p, o)\n        if key not in unique or score > unique[key][\'score\']:\n            unique[key] = {\'subject_index\': s, \'object_index\': o, \'subject_instance\': objects[s][\'instance_id\'],\n                           \'object_instance\': objects[o][\'instance_id\'], \'subject\': \'person\', \'object\': objects[o][\'label\'],\n                           \'predicate\': p, \'score\': score, \'threshold\': thresholds[p],\n                           \'text\': relation_text(p, objects[o][\'label\'])}\n    # Margin is only a fixed cap ordering; not calibrated cross-predicate confidence.\n    accepted = sorted((c for c in unique.values() if c[\'score\'] >= c[\'threshold\']),\n                      key=lambda c: (-(c[\'score\'] - c[\'threshold\']), c[\'subject_index\'], c[\'predicate\'], c[\'object_index\']))\n    selected = accepted[:max_relations]\n    endpoints = {c[k] for c in selected for k in (\'subject_index\', \'object_index\')}\n    kept = [d for d in objects if d[\'confidence\'] >= object_threshold or d[\'index\'] in endpoints]\n    names = list(dict.fromkeys(d[\'label\'] for d in sorted(kept, key=lambda d: (-d[\'confidence\'], d[\'index\']))))[:max_object_names]\n    object_prompt = \'a photo containing \' + \', \'.join(noun(n) for n in names) + \'.\' if names else \'\'\n    texts = list(dict.fromkeys(c[\'text\'] for c in selected))\n    relation_prompt = \'a photo showing \' + \'; \'.join(texts) + \'.\' if texts else \'\'\n    previews = sorted(unique.values(), key=lambda c: (-c[\'score\'], c[\'subject_index\'], c[\'predicate\'], c[\'object_index\']))[:3]\n    return {\'objects\': kept, \'relations\': selected, \'object_prompt\': object_prompt,\n            \'relation_prompt\': relation_prompt, \'prompt\': relation_prompt or object_prompt or \'a photo.\',\n            \'mode\': \'object_relation\' if selected else \'object_only\' if names else \'no_objects\',\n            \'above_threshold_claims\': len(accepted), \'capped_claims\': len(accepted) - len(selected),\n            \'candidate_preview\': [{**c, \'passes_locked_threshold\': c[\'score\'] >= c[\'threshold\']} for c in previews]}\n', 'captioning/upt_action_text.py': '"""Explicit text for all 116 positive HICO action labels; no new action inference."""\n\nACTION_TEXT = dict(line.split(\'=\', 1) for line in \'\'\'adjust=adjusting\nassemble=assembling\nblock=blocking\nblow=blowing toward\nboard=boarding\nbreak=breaking\nbrush_with=brushing with\nbuy=buying\ncarry=carrying\ncatch=catching\nchase=chasing\ncheck=checking\nclean=cleaning\ncontrol=controlling\ncook=cooking\ncut=cutting\ncut_with=cutting with\ndirect=directing\ndrag=dragging\ndribble=dribbling\ndrink_with=drinking from\ndrive=driving\ndry=drying\neat=eating\neat_at=eating at\nexit=exiting\nfeed=feeding\nfill=filling\nflip=flipping\nflush=flushing\nfly=flying\ngreet=greeting\ngrind=grinding on\ngroom=grooming\nherd=herding\nhit=hitting\nhold=holding\nhop_on=hopping on\nhose=hosing\nhug=hugging\nhunt=hunting\ninspect=inspecting\ninstall=installing\njump=jumping\nkick=kicking\nkiss=kissing\nlasso=lassoing\nlaunch=launching\nlick=licking\nlie_on=lying on\nlift=lifting\nlight=lighting\nload=loading\nlose=losing\nmake=making\nmilk=milking\nmove=moving\nopen=opening\noperate=operating\npack=packing\npaint=painting\npark=parking\npay=paying at\npeel=peeling\npet=petting\npick=picking\npick_up=picking up\npoint=pointing\npour=pouring from\npull=pulling\npush=pushing\nrace=racing\nread=reading\nrelease=releasing\nrepair=repairing\nride=riding\nrow=rowing\nrun=running\nsail=sailing\nscratch=scratching\nserve=serving\nset=setting\nshear=shearing\nsign=signing\nsip=sipping from\nsit_at=sitting at\nsit_on=sitting on\nslide=sliding on\nsmell=smelling\nspin=spinning\nsqueeze=squeezing\nstab=stabbing\nstand_on=standing on\nstand_under=standing under\nstick=sticking\nstir=stirring\nstop_at=stopping at\nstraddle=straddling\nswing=swinging\ntag=tagging\ntalk_on=talking on\nteach=teaching\ntext_on=texting on\nthrow=throwing\ntie=tying\ntoast=toasting with\ntrain=training\nturn=turning\ntype_on=typing on\nwalk=walking\nwash=washing\nwatch=watching\nwave=waving at\nwear=wearing\nwield=wielding\nzip=zipping\'\'\'.splitlines())\n\nassert len(ACTION_TEXT) == 116 and \'no_interaction\' not in ACTION_TEXT\n\n\ndef action_predicate(verb):\n    return {\'hold\': \'holding\', \'ride\': \'riding\'}.get(verb, verb)\n\n\ndef uniform_policy(threshold=.5):\n    import math\n    if not math.isfinite(threshold) or not 0 < threshold <= 1:\n        raise ValueError(\'Exploratory threshold must be in (0,1].\')\n    return {\'mode\': \'all_actions_exploratory\', \'calibrated\': False,\n            \'predicates\': {action_predicate(v): {\'threshold\': threshold} for v in ACTION_TEXT}}\n', 'captioning/selected_relation_prompts.py': '"""Select scored detector claims without consulting annotation IDs or GT labels."""\nimport hashlib\nimport json\nimport math\nfrom collections import defaultdict\nfrom captioning.supervised_relations import RELATIONS\n\n\ndef entity_id(filename, label, box):\n    payload = json.dumps([filename, label, box], separators=(\',\', \':\'))\n    return \'det_\' + hashlib.sha256(payload.encode()).hexdigest()[:12]\n\n\ndef phrase(label):\n    article = \'an\' if label[:1].lower() in \'aeiou\' else \'a\'\n    return f\'{article} {label}\'\n\n\ndef build_selected_prompts(scored_pairs, thresholds, max_relations=3, object_inventory=None):\n    if len(thresholds) != len(RELATIONS) or max_relations < 1:\n        raise ValueError(\'Invalid thresholds/relation limit.\')\n    if not all(math.isfinite(x) for x in thresholds):\n        raise ValueError(\'Thresholds must be finite.\')\n    groups = defaultdict(list)\n    for filename in object_inventory or {}:\n        groups[filename] = []\n    for row in scored_pairs:\n        if len(row[\'scores\']) != len(RELATIONS) or not all(math.isfinite(v) for v in row[\'scores\']):\n            raise ValueError(\'Invalid scores.\')\n        for role in (\'subject\', \'object\'):\n            box = row[role+\'_box\']\n            if (len(box) != 4 or not all(math.isfinite(v) for v in box)\n                    or not 0 <= box[0] < box[2] <= 1 or not 0 <= box[1] < box[3] <= 1):\n                raise ValueError(\'Expected normalized nonempty detector boxes.\')\n        groups[row[\'filename\']].append(row)\n    result = {}\n    for filename, rows in sorted(groups.items()):\n        objects, candidates = {}, {}\n        for entity in (object_inventory or {}).get(filename, []):\n            identity = entity_id(filename, entity[\'label\'], entity[\'bbox\'])\n            objects[identity] = {**entity, \'instance_id\': identity,\n                                 \'prompt\': \'a photo of \'+phrase(entity[\'label\'])}\n        for row in rows:\n            ids = []\n            for role in (\'subject\', \'object\'):\n                box, label = row[role+\'_box\'], row[role]\n                identity = entity_id(filename, label, box)\n                ids.append(identity)\n                objects[identity] = {\'instance_id\': identity, \'label\': label, \'bbox\': box,\n                    \'confidence\': row.get(role+\'_detector_confidence\'), \'prompt\': \'a photo of \'+phrase(label)}\n            for c, relation in enumerate(RELATIONS):\n                score = row[\'scores\'][c]\n                if score < thresholds[c] or ids[0] == ids[1]:\n                    continue\n                key = (ids[0], relation, ids[1])\n                claim = {\'subject_instance\': ids[0], \'subject\': row[\'subject\'],\n                         \'object_instance\': ids[1], \'object\': row[\'object\'], \'relation\': relation,\n                         \'score\': score, \'threshold\': thresholds[c], \'score_margin\': score-thresholds[c],\n                         \'prompt\': f\'a photo of {phrase(row["subject"])} {relation.replace("_", " ")} {phrase(row["object"])}\'}\n                if key not in candidates or score > candidates[key][\'score\']:\n                    candidates[key] = claim\n        # Margin is a deterministic cap priority, NOT calibrated cross-class confidence.\n        ordered = sorted(candidates.values(), key=lambda c: (-c[\'score_margin\'], c[\'subject_instance\'],\n                                                            c[\'relation\'], c[\'object_instance\']))\n        selected = ordered[:max_relations]\n        labels = sorted({entity[\'label\'] for entity in objects.values()})\n        object_text = \'a photo of \'+\', \'.join(phrase(label) for label in labels) if labels else \'\'\n        relation_text = (\'a photo showing \'+\', \'.join(\n            f\'{phrase(c["subject"])} {c["relation"].replace("_", " ")} {phrase(c["object"])}\' for c in selected))\n        result[filename] = {\'objects\': list(objects.values()), \'relations\': selected,\n                            \'object_prompt\': object_text,\n                            \'relation_prompt\': relation_text if selected else \'\',\n                            \'prompt\': relation_text if selected else object_text,\n                            \'mode\': \'object_relation\' if selected else \'object_only\' if objects else \'no_objects\',\n                            \'above_threshold_claims\': len(ordered), \'capped_claims\': len(ordered)-len(selected)}\n    return result\n', 'captioning/supervised_relations.py': '"""V-COCO pair supervision. Missing links are UNKNOWN, never invented negatives."""\nimport math\nimport random\nfrom collections import Counter, defaultdict\n\nRELATION_SPECS = (\n    (\'hold\', \'obj\', \'holding\'), (\'ride\', \'instr\', \'riding\'),\n    (\'sit\', \'instr\', \'sitting_on\'), (\'carry\', \'obj\', \'carrying\'),\n    (\'eat\', \'obj\', \'eating\'), (\'talk_on_phone\', \'instr\', \'talking_on\'),\n)\nRELATIONS = [spec[2] for spec in RELATION_SPECS]\n\n\ndef decode_action(action):\n    """Official serialization stores all agents, then each role column."""\n    n = len(action[\'ann_id\'])\n    roles = action[\'role_name\']\n    flat = action[\'role_object_id\']\n    if n == 0 or len(flat) != n * len(roles):\n        raise ValueError(\'Invalid V-COCO role array.\')\n    if len(action[\'image_id\']) != n or len(action[\'label\']) != n:\n        raise ValueError(\'V-COCO column lengths differ.\')\n    if roles[0] != \'agent\' or flat[:n] != action[\'ann_id\']:\n        raise ValueError(\'Agent column does not match annotation IDs.\')\n    rows = {}\n    for i, sid in enumerate(action[\'ann_id\']):\n        if sid in rows or action[\'label\'][i] not in (-1, 0, 1):\n            raise ValueError(\'Duplicate actor or invalid action label.\')\n        rows[sid] = {\'image_id\': action[\'image_id\'][i], \'label\': action[\'label\'][i],\n                     \'roles\': {role: flat[j * n + i] for j, role in enumerate(roles)}}\n    return rows\n\n\ndef normalized_box(annotation, image):\n    x, y, w, h = annotation[\'bbox\']\n    width, height = image[\'width\'], image[\'height\']\n    if not all(math.isfinite(v) for v in (x, y, w, h, width, height)):\n        raise ValueError(\'Non-finite box.\')\n    if min(w, h, width, height) <= 0:\n        raise ValueError(\'Empty box/image.\')\n    box = [max(0, x / width), max(0, y / height),\n           min(1, (x + w) / width), min(1, (y + h) / height)]\n    if not (box[0] < box[2] and box[1] < box[3]):\n        raise ValueError(\'Box outside image.\')\n    return box\n\n\ndef build_pairs(actions, instances, caption_images, max_objects=12):\n    if max_objects < 1:\n        raise ValueError(\'max_objects must be positive.\')\n    by_action = {a[\'action_name\']: decode_action(a) for a in actions}\n    if any(name not in by_action for name, _, _ in RELATION_SPECS):\n        raise ValueError(\'Required V-COCO action missing.\')\n    images = {x[\'id\']: x for x in instances[\'images\']}\n    anns = {x[\'id\']: x for x in instances[\'annotations\']}\n    categories = {x[\'id\']: x[\'name\'] for x in instances[\'categories\']}\n    caption = {x[\'cocoid\']: x for x in caption_images}\n    if len(caption) != len(caption_images):\n        raise ValueError(\'Duplicate caption image ID.\')\n    by_image = defaultdict(list)\n    boxes = {}\n    for aid, ann in anns.items():\n        if ann.get(\'iscrowd\', 0):\n            continue\n        boxes[aid] = normalized_box(ann, images[ann[\'image_id\']])\n        by_image[ann[\'image_id\']].append(aid)\n    subjects = sorted(set(s for a in by_action.values() for s in a))\n    rows, stats = [], Counter()\n    for sid in subjects:\n        if sid not in anns:\n            raise ValueError(f\'Missing actor annotation {sid}.\')\n        ann = anns[sid]\n        iid = ann[\'image_id\']\n        if iid not in caption:\n            raise ValueError(f\'Image {iid} missing from caption split metadata.\')\n        if caption[iid][\'split\'] not in {\'train\', \'restval\'}:\n            stats[\'excluded_caption_heldout_subjects\'] += 1\n            continue\n        if sid not in boxes:\n            stats[\'excluded_crowd_subjects\'] += 1\n            continue\n        if categories[ann[\'category_id\']] != \'person\':\n            raise ValueError(\'V-COCO agent must be a person.\')\n        positives = set()\n        for name, role, _ in RELATION_SPECS:\n            item = by_action[name].get(sid)\n            if item and item[\'image_id\'] != iid:\n                raise ValueError(\'Actor/image ID mismatch.\')\n            if item and item[\'label\'] == 1:\n                oid = item[\'roles\'].get(role, 0)\n                if oid:\n                    if oid not in anns or anns[oid][\'image_id\'] != iid or oid == sid:\n                        raise ValueError(\'Invalid positive role object.\')\n                    if oid in boxes:\n                        positives.add(oid)\n                    else:\n                        stats[\'unusable_positive_box\'] += 1\n                else:\n                    stats[\'positive_action_without_visible_role\'] += 1\n        # Keep every positive; fill with deterministic same-image candidates.\n        candidates = sorted(set(by_image[iid]) - {sid} - positives)\n        candidates = sorted(positives) + candidates[:max(0, max_objects - len(positives))]\n        for oid in candidates:\n            labels = []\n            for name, role, _ in RELATION_SPECS:\n                item = by_action[name].get(sid)\n                label = -1\n                if item and item[\'label\'] == 0:\n                    label = 0  # Annotated absence of this actor action implies absence for any B.\n                elif item and item[\'label\'] == 1 and item[\'roles\'].get(role, 0) == oid:\n                    label = 1\n                # label=1 with another B, missing role, or label=-1 remains UNKNOWN.\n                labels.append(label)\n            if all(v == -1 for v in labels):\n                continue\n            image = caption[iid]\n            rows.append({\'image_id\': iid, \'filename\': image[\'filename\'],\n                         \'filepath\': image.get(\'filepath\', \'\'), \'subject_id\': sid,\n                         \'object_id\': oid, \'subject\': \'person\',\n                         \'object\': categories[anns[oid][\'category_id\']],\n                         \'subject_box\': boxes[sid], \'object_box\': boxes[oid],\n                         \'labels\': labels, \'caption_split\': image[\'split\']})\n    return rows, dict(stats)\n\n\ndef diagnostic_subset(rows, limit, seed=42):\n    if limit < 1:\n        raise ValueError(\'Pair limit must be positive.\')\n    rng = random.Random(seed)\n    pos = [r for r in rows if 1 in r[\'labels\']]\n    neg = [r for r in rows if 1 not in r[\'labels\']]\n    rng.shuffle(pos)\n    rng.shuffle(neg)\n    by_target = defaultdict(list)\n    for row in rows:\n        by_target[(row[\'image_id\'], row[\'object_id\'])].append(row)\n    # Enrich a bounded portion with confirmed actor-counterpart anchors; this is\n    # explicitly a development diagnostic, not a prevalence-preserving benchmark.\n    actor_anchors = [row for row in pos if any(\n        other[\'subject_id\'] != row[\'subject_id\']\n        and any(a == 1 and b == 0 for a, b in zip(row[\'labels\'], other[\'labels\']))\n        for other in by_target[(row[\'image_id\'], row[\'object_id\'])])]\n    chosen = actor_anchors[:limit // 8]\n    used = {(r[\'subject_id\'], r[\'object_id\']) for r in chosen}\n    chosen += [r for r in pos if (r[\'subject_id\'], r[\'object_id\']) not in used][\n        :max(0, min(len(pos), limit // 2) - len(chosen))]\n    used = {(r[\'subject_id\'], r[\'object_id\']) for r in chosen}\n    for anchor in list(chosen):\n        candidates = by_target[(anchor[\'image_id\'], anchor[\'object_id\'])]\n        for other in candidates:\n            key = (other[\'subject_id\'], other[\'object_id\'])\n            if (key not in used and other[\'subject_id\'] != anchor[\'subject_id\']\n                    and any(a == 1 and b == 0 for a, b in zip(anchor[\'labels\'], other[\'labels\']))):\n                chosen.append(other)\n                used.add(key)\n                break\n        if len(chosen) >= limit:\n            break\n    # Fill remaining slots with all-zero/unknown pairs before more positives.\n    # Positive-first fill previously skewed the intended diagnostic composition.\n    remaining = [r for r in neg + pos if (r[\'subject_id\'], r[\'object_id\']) not in used]\n    chosen += remaining[:max(0, limit - len(chosen))]\n    return sorted(chosen, key=lambda r: (r[\'image_id\'], r[\'subject_id\'], r[\'object_id\']))\n\n\ndef contrast_edges(rows, max_per_positive=4):\n    """Only annotated-zero claims can be negative; no role swaps inferred false."""\n    by_target = defaultdict(list)\n    for i, row in enumerate(rows):\n        by_target[(row[\'image_id\'], row[\'object_id\'])].append(i)\n    edges = []\n    for i, row in enumerate(rows):\n        for r, label in enumerate(row[\'labels\']):\n            if label != 1:\n                continue\n            wrong_pred = [(i, c, \'predicate\') for c, v in enumerate(row[\'labels\']) if v == 0]\n            wrong_actor = [(j, r, \'actor\') for j in by_target[(row[\'image_id\'], row[\'object_id\'])]\n                           if rows[j][\'subject_id\'] != row[\'subject_id\'] and rows[j][\'labels\'][r] == 0]\n            # Alternate types so a predicate-heavy anchor does not exclude all actor edges.\n            negatives = []\n            while wrong_pred or wrong_actor:\n                if wrong_actor:\n                    negatives.append(wrong_actor.pop(0))\n                if wrong_pred:\n                    negatives.append(wrong_pred.pop(0))\n            edges += [(i, r, j, c, kind) for j, c, kind in negatives[:max_per_positive]]\n    return edges\n\n\ndef pair_statistics(rows):\n    return {\'pairs\': len(rows), \'images\': len({r[\'image_id\'] for r in rows}),\n            \'labels\': {name: dict(Counter(str(r[\'labels\'][c]) for r in rows))\n                       for c, name in enumerate(RELATIONS)},\n            \'contrast_edges\': dict(Counter(e[-1] for e in contrast_edges(rows)))}\n\n\ndef select_unseen_pairs(rows, seen, count=500, seed=20261013):\n    eligible = [r for r in rows if r[\'image_id\'] not in seen]\n    selected = diagnostic_subset(eligible, count, seed=seed)\n    if len(selected) != count:\n        raise ValueError(\'Insufficient new pairs; do not silently change protocol.\')\n    if {r[\'image_id\'] for r in selected} & set(seen):\n        raise ValueError(\'Seen image selected.\')\n    return selected, eligible\n', 'detector_index.py': "from pathlib import Path\nimport numpy as np\ndef detection_index(source):\n    if isinstance(source, dict) and 'data' in source:\n        source = source['data']\n    if not isinstance(source, dict):\n        raise ValueError('Detector source must map filenames to records')\n    result = {}\n    for name, record in source.items():\n        filename = Path(name).name\n        if filename in result:\n            raise ValueError('Duplicate detector basename: ' + filename)\n        objects = record if isinstance(record, list) else record['objects']\n        if not isinstance(objects, list):\n            raise ValueError('Missing/malformed objects: ' + filename)\n        for obj in objects:\n            box = np.asarray(obj['bbox'], dtype=float)\n            score = float(obj['conf'])\n            if box.shape != (4,) or not np.isfinite(box).all() or (not np.isfinite(score)) or (not 0 <= score <= 1):\n                raise ValueError('Nonfinite/invalid detector value: ' + filename)\n            if box[2] <= box[0] or box[3] <= box[1]:\n                raise ValueError('Nonpositive xyxy area: ' + filename)\n        result[filename] = objects\n    return result"}
ROI_PINS={'val_0000.npz': 'ae4b911eb0e4e80868474eeb53220564c926030004e25da7b2fdf66be4cb3113', 'val_0000.json': '7a13112d6b180dea9fdedbf405aca42e64564310977c105f31fe5edced4a93f4', 'val_0001.npz': '38ef107b5cfe9b9febf974f46ea137370f1510f7e5fdaf24bb69cdf952a302e0', 'val_0001.json': 'a868d753742411bed0554747eae20f0131b9df3d6c158f21372a7252c4928eaf', 'val_0002.npz': '2dfec7f2b16f9cf0db30660f7eb9de60fea133b44d331e83daaa2b0d37117680', 'val_0002.json': '4bed69399d25af1eb8bf52e61972ed162188ae2b2c806c5616894952712c86b6', 'val_0003.npz': '4b45856f9f21b494146a8950a81799c0add568c7808c7a4a70c8ff85501428d6', 'val_0003.json': 'afb5f12cf003bd7fa6fda882c0407e0177abc2b848cd597e6a8928aed3022f27', 'val_0004.npz': 'c869c86e9668dad788f6790cd954a1373db3193341f88666c7fa393ebd1c4387', 'val_0004.json': '3a500b41aeb9e471d6f339456a96bc1be1f5a8903bd90d1f8557e34cc404efb0', 'manifest.json': '512bf34336be54f2854743bd06004dbc30ff506cda130c03378d77a6d3cf6130'}
WANTED=[4375, 5282, 13004, 13031, 15303, 16546, 18193, 22420, 25005, 29573, 30347, 31113, 36032, 36607, 46101, 48554, 55167, 57265, 58225, 63549, 66485, 73927, 76822, 80895, 82396, 82986, 83573, 83925, 94607, 105973, 111909, 118740, 122542, 122766, 124116, 131869, 133620, 134103, 135155, 142687, 154816, 161187, 161978, 162580, 163112, 163543, 166624, 167656, 189248, 197997, 199017, 199977, 202228, 204757, 207561, 210394, 214604, 217622, 226008, 231079, 236166, 259690, 263251, 266491, 270097, 271681, 276693, 294537, 296825, 304596, 308362, 310369, 314493, 314757, 314996, 317188, 317595, 322145, 327656, 328791, 328921, 334309, 336431, 352729, 365129, 366822, 374829, 376206, 376368, 377715, 377999, 379048, 384635, 389123, 391463, 409241, 409346, 425676, 434993, 435309, 438907, 439522, 442417, 461262, 463722, 467580, 467791, 472878, 481670, 492677, 497855, 499400, 505099, 511379, 515444, 525439, 528276, 538414, 543402, 548209, 548550, 557596, 561745, 562994, 563885, 564355, 570594, 571384, 574108, 575349]

from collections import Counter
from pathlib import Path
import hashlib, json, tempfile, sys
import numpy as np

DATASETS = Path('/kaggle/input/datasets')
def checked(root, name, digest):
    files = list(root.rglob(name))
    if len(files) != 1:
        raise ValueError(f'Expected one {name} under {root}')
    with files[0].open('rb') as stream:
        if hashlib.file_digest(stream, 'sha256').hexdigest() != digest:
            raise ValueError(f'Unexpected SHA256: {files[0]}')
    return files[0]

work = Path(tempfile.mkdtemp(prefix='relation-join-'))
for name, text in SOURCES.items():
    path = work/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
(work/'captioning/__init__.py').touch()
sys.path.insert(0,str(work))
from prompt_adapter_data import detector_regions, join_entry, label, overlap, IOU, PROMPT_SHA
from detector_index import detection_index

def read(path):
    return json.loads(path.read_text(encoding='utf-8'))

upt = read(checked(DATASETS/'ducanh2403/upt-prompts-full-all-actions-v2','prompt_cache.json',PROMPT_SHA))
yolo = detection_index(read(checked(DATASETS/'ducanh2403/objectdetectionecache','objectdetectioncache.json','498825daf8f8773a1c92f4111dc4ccf1f4296343f952b486e0aa56caf82ec04c')))
links = checked(DATASETS/'ducanh2403/csa-prompt-full-karpathy-v175','prompt_links.jsonl','e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0')
with links.open(encoding='utf-8') as stream:
    expected = {r['coco_id']:r for r in map(json.loads,stream) if r['group']=='val' and r['coco_id'] in WANTED}
roi = DATASETS/'ducanh2403/csa-full-roi-karpathy-v134'
manifest = read(checked(roi,'manifest.json',ROI_PINS['manifest.json']))
def trace(objects, regions):
    reasons = {}
    for obj in objects:
        candidates = [r for r in regions if r['label']==label(obj['label'])]
        values = overlap(obj['bbox'],[r['bbox'] for r in candidates])
        reason, best, target = 'no_same_class_roi', None, None
        if len(values):
            best = float(values.max())
            tied = np.flatnonzero(np.isclose(values,best,rtol=0,atol=1e-8))
            if best<IOU:
                reason='below_iou'
            elif len(tied)!=1:
                reason='ambiguous_tie'
            else:
                reason='matched'
                target=candidates[int(tied[0])]['roi']
        reasons[obj['instance_id']]=dict(reason=reason,best_iou=best,roi=target,same_class_rois=len(candidates),label=obj['label'])
    occupancy=Counter(v['roi'] for v in reasons.values() if v['reason']=='matched')
    for item in reasons.values():
        if item['reason']=='matched' and occupancy[item['roi']]>1:
            item['reason']='collision'
    return reasons

def key(r):
    return (r['subject_instance'],r['predicate'],r['object_instance'])

def proposal(rel, reasons, objects, regions):
    subject_id, object_id = rel['subject_instance'], rel['object_instance']
    subject, target = reasons[subject_id], reasons[object_id]
    best = target['best_iou']
    if subject['reason'] != 'matched' or target['reason'] != 'below_iou' or not .45 <= best < IOU:
        return None, None
    obj = objects[object_id]
    candidates = [r for r in regions if r['label'] == target['label'] or r['label'] == label(target['label'])]
    values = overlap(obj['bbox'], [r['bbox'] for r in candidates])
    top = np.flatnonzero(np.isclose(values,best,rtol=0,atol=1e-8))
    if len(top) != 1:
        return None, 'tied_best'
    roi = candidates[int(top[0])]
    competitors = [o['bbox'] for k,o in objects.items() if k != object_id and label(o['label']) == roi['label']]
    if any(overlap(roi['bbox'], competitors) >= .45):
        return None, 'contested_roi'
    person_roi = next(r for r in regions if r['roi'] == subject['roi'])
    return dict(subject_roi=person_roi['bbox'], upt_object=obj['bbox'], yolo_object=roi['bbox'],
                object_roi=roi['roi'], iou=best, predicate=rel['predicate'], object_instance=object_id), 'proposed'

rows=[]
proposed=[]
screen=Counter()
seen=set()
totals={role:Counter() for role in ('subject','object')}
for shard in manifest['shards']:
    if shard['group']!='val':
        continue
    names=[n for n in shard['files'] if n in ROI_PINS]
    meta,= [n for n in names if n.endswith('.json')]
    array,= [n for n in names if n.endswith('.npz')]
    images=read(checked(roi,meta,ROI_PINS[meta]))['images']
    with np.load(checked(roi,array,ROI_PINS[array]),allow_pickle=False) as payload:
        ids, offsets, boxes=(payload[n] for n in ('ids','offsets','boxes'))
        for i,m in enumerate(images):
            cid=m['coco_id']
            if cid not in expected:
                continue
            if cid in seen or int(ids[i])!=cid or m['group']!='val' or m['source_split']!='val':
                raise ValueError(f'Unexpected ROI row {cid}')
            seen.add(cid)
            name=m['filename']
            regions=detector_regions(yolo[name],m['width'],m['height'],boxes[int(offsets[i]):int(offsets[i+1])],m['region_kinds'])
            entry=upt[name]
            replay=join_entry(entry,regions)
            saved=expected[cid]
            if any(replay[k]!=saved[k] for k in ('counts','relations','objects')):
                raise ValueError(f'v175 mapping replay differs for {cid}')
            reasons=trace(entry['objects'],regions)
            objects={o['instance_id']:o for o in entry['objects']}
            kept={key(r) for r in replay['relations']}
            for rel in entry['relations']:
                if key(rel) in kept:
                    continue
                row=dict(coco_id=cid,filename=name,relation=rel['text'])
                for role in ('subject','object'):
                    detail=reasons[rel[role+'_instance']]
                    row[role]=detail
                    totals[role][detail['reason']]+=1
                if row['subject']['reason']==row['object']['reason']=='matched':
                    raise ValueError(f'Unexplained relation loss for {cid}')
                rows.append(row)
                pick, status = proposal(rel, reasons, objects, regions)
                if status:
                    screen[status] += 1
                if pick:
                    proposed.append(dict(coco_id=cid, filename=name, text=rel['text'], **pick))
if seen!=set(WANTED) or len(rows)!=207:
    raise ValueError('Incomplete lost relation coverage')
joint=Counter(f"{r['subject']['reason']} / {r['object']['reason']}" for r in rows)

if sum(screen.values()) != 37:
    raise ValueError('Unexpected near-threshold cohort')
IMAGE_SHA={4375: '999c2ccf98b2798b57175a0e75d72fe856b5d05d8d1f8d3f992ec79f04a2d71d', 15303: '0937e550afd880c8478e0b3667317ffffaae53329e8b3be532b7e6ebd559c9d9', 25005: '4503c81279480d339c13f79d64a63609fc50455605a3956d332965c140a80658', 29573: '7da3cca824b4cddd6212479479d4e62ed03329e9267096aab3d211b4a7604f29', 30347: '1136e04bdc8a729ba237ee8058ee5678bf4ade1d80ab134285d987b3b9e9b5d4', 48554: '859837b4332079677cd95f952be05bd1026488cb165772093427af0ee78d3c4a', 57265: '7f7156c13ef7fc450bb19ca960f5ad8597b59033535e161ddc713f0de1a6e91c', 63549: '68e700a4a5c7c3824b68d6c1a848b5f26f2615a3f7911b145b51ec5c9e323212', 73927: '47a9ebf3343822f0a0b1cf985b54e0ba4de9cc67e07aa8a13f1e2467aac99d49', 80895: 'ccccd327374176e5b9867eb5502e306c17abb1c082e330dbefbbf3ded6e612db', 122542: '622083b6ed4285f7e32571d9bc660ecaef750412ec09a8f75ecef1c132412b46', 122766: '094754a4664099cc7a6fc12465d18c840c666d38104c15396ccba161a7e0247b', 131869: 'f8145242fc1fd0e9f64fc4abf31bdf3d84e464367968a2a093a22fdd2787d323', 135155: '077d6ea1f51c110ef8429f5dadeef45213a513fb7f2b9c4d5e808228a36a4399', 217622: 'ba17708fd73b1e4286c2a24fb9647ab97c5d5d44d568b5e9c5f710b45dc8d643', 294537: '1863125b46e51a5222f66976779315c8b9856f465f08fc81903c6983cc17dbc7', 336431: '4c17b31e2ecc89d08d97cf856fd8b725e361693ff649e5f0111b82df5c00434b', 352729: '3987d705bfc9b23d9e6f2f44504124b1097f4af061db6407e9da32229e2cd2ef', 366822: '7fb7cffcf05cd409b74bf6faaeb2fac3a2db9ec4ed46fa332f43d02f716a744f', 481670: '1926544f88dae8a09a3789fbe0d5e78605192824a79cd8254a768119674b24b7', 528276: 'e3477388ea64d498521cbd71ac751784c307c29752f662208b1794e83df7b4f8', 557596: 'bd666229b89cf45adb02d25ed48fbc475b28e4ce0813c1e21bda22b0a81d43d5'}

import base64, html, zipfile
coco = DATASETS/'vuthetam/mscoco-2014'
output = Path('/kaggle/working')
report = output/'csa_relation_fallback_valid.json'
report.write_text(json.dumps(dict(screen=screen, proposals=proposed),indent=2),encoding='utf-8')
needed = {r['filename'] for r in proposed}
images = {}
if needed:
    for path in coco.rglob('*.jpg'):
        if path.name in needed:
            if path.name in images:
                raise ValueError(f'Ambiguous image {path.name}')
            images[path.name] = path
if set(images) != needed:
    raise ValueError('Attach mscoco-2014 for the review images')
cards = []
for i,r in enumerate(proposed,1):
    raw = images[r['filename']].read_bytes()
    if hashlib.sha256(raw).hexdigest() != IMAGE_SHA[r['coco_id']]:
        raise ValueError(f"Image SHA differs for {r['coco_id']}")
    encoded = base64.b64encode(raw).decode()
    rectangles = []
    for key,color in [('subject_roi','lime'),('upt_object','orange'),('yolo_object','deepskyblue')]:
        x,y,x2,y2 = r[key]
        rectangles.append(f'<rect x="{x*100}" y="{y*100}" width="{(x2-x)*100}" height="{(y2-y)*100}" fill="none" stroke="{color}" stroke-width=".6"/>')
    select = '<select><option value="">Unreviewed</option><option value="yes">Yes</option><option value="no">No</option><option value="unclear">Unclear</option></select>'
    cards.append(f'<article data-case="{i}"><h2>#{i} · COCO {r["coco_id"]} · IoU {r["iou"]:.3f}</h2>'
                 f'<p>{html.escape(r["text"])}</p><div><img src="data:image/jpeg;base64,{encoded}">'
                 f'<svg viewBox="0 0 100 100" preserveAspectRatio="none">{"".join(rectangles)}</svg></div>'
                 f'<p>Same object in orange and blue? {select}</p><p>Is the relation true for the green person and blue object? {select}</p></article>')
page = output/'csa_relation_fallback_review.html'
page.write_text('''<!doctype html><meta charset="utf-8"><title>Relation fallback review</title>
<style>body{font:16px system-ui;max-width:900px;margin:24px auto;padding:0 15px}article{border-top:1px solid #aaa;padding:16px 0}article div{position:relative;width:fit-content;max-width:100%}img{display:block;max-width:100%;max-height:500px}svg{position:absolute;inset:0;width:100%;height:100%;pointer-events:none}</style>
<h1>Relation fallback review</h1><p>Green: matched person. Orange: UPT object. Blue: proposed YOLO object.</p>
<button id="save">Export labels</button>'''+''.join(cards)+'''<script>
document.getElementById('save').onclick=()=>{
 const rows=[...document.querySelectorAll('article')].map(card=>{
  const [same_object,relation_correct]=[...card.querySelectorAll('select')].map(s=>s.value);
  return {case_id:Number(card.dataset.case),same_object,relation_correct};
 }).filter(r=>r.same_object||r.relation_correct);
 const url=URL.createObjectURL(new Blob([JSON.stringify(rows,null,2)],{type:'application/json'}));
 const link=document.createElement('a'); link.href=url; link.download='csa_relation_fallback_labels.json';link.click();
 setTimeout(()=>URL.revokeObjectURL(url),1000);
};
</script>''',encoding='utf-8')
archive=output/'csa_relation_fallback_valid_review.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in (report,page):
        z.write(path,path.name)
print(dict(screen),archive,flush=True)
